# Tracing the Spectrum

The spectrum of a point source on a long-slit frame is a narrow horizontal ridge: bright along
the dispersion axis, only a few pixels wide across the slit. Everything the rest of this book
does, from measuring the sky and collapsing the image to one dimension to deciding which pixels
belong to the object at all, needs to know *where that ridge is*, column by column.

That path is the **trace**, and finding it is the first real reduction step. `specreduce.tracing` offers three `Trace` classes for doing this, and the choice between them is really a choice about
how much you are willing to assume:

| Class | Assumption | Use when |
|---|---|---|
| `FlatTrace` | the trace is a horizontal line | the instrument is well aligned, or you need a quick look |
| `FitTrace` | the trace is a smooth low-order curve | almost always; this is the workhorse |
| `ArrayTrace` | nothing; you supply the positions | the shape defeats a polynomial, or the positions come from elsewhere |

This chapter works through all three on the science frame, and, more importantly, shows how
to tell whether the trace you got is any good. There is no ground truth here, so "good" has to
mean *internally consistent*: the trace stops changing when the model is given more freedom,
different peak-finding methods agree on it, and it follows the ridge when drawn over the image.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from astropy.visualization import simple_norm
from specreduce.tracing import ArrayTrace, FitTrace, FlatTrace

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)
print(f"science frame: {ny} cross-dispersion x {nx} dispersion pixels")

## Finding the source

Before fitting anything, look at where the light is. This is the full science frame, drawn with
the same logarithmic stretch as in Chapter 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ax.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
)
ax.set(xlabel="Dispersion [pix]", ylabel="Cross-dispersion [pix]");

The bright horizontal ridge is TrES-3, and the fainter ridge above it is a second star that fell
in the slit. At this scale both look perfectly straight, which is exactly the impression a trace
has to correct. A better first view is to collapse the frame along the dispersion axis, which
turns the 2D image into a rough estimate of the **cross-dispersion profile**: rough, because the
collapse ignores the curvature of the trace and smears a tilted or curved trace over several
rows. Even so, a point source appears as a spike sitting on the broad plateau of the sky.

In [ ]:
profile = obj.data.mean(axis=1)
sky_level = np.median(profile)

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(rows, profile, lw=0.8)
ax.axhline(sky_level, c="0.6", lw=0.8, ls="--")
ax.text(
    365,
    sky_level * 1.15,
    f"sky $\\approx$ {sky_level:.0f} e$^-$",
    ha="right",
    size=8,
    c="0.4",
)

for r, name in ((135, "target"), (311, "faint star")):
    ax.annotate(
        f"{name}\nrow {r}",
        (r, profile[r]),
        (r + 12, profile[r]),
        size=8,
        va="center",
        arrowprops=dict(arrowstyle="-", lw=0.6, color="0.4"),
    )

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    xlabel="Cross-dispersion [pix]",
    ylabel="Mean signal [e$^-$]",
);

Four sources stand out. The tall spike at **row 135** is TrES-3 itself, about fifty times the sky
level. The smaller bump at **row 311** is a second star that happened to fall in the
40-arcsecond slit. It peaks at roughly three percent of the target, and we will come back to
it, because faint neighbors are exactly what breaks naive tracing. Two fainter sources remain:
one blended into the wing of TrES-3, and a fainter one still near row 40. This tutorial ignores
both.

The spike is narrow: its full width at half maximum is about six pixels, the seeing disk as
sampled by this binning. Bear in mind that this is the width of the *averaged* profile. The
collapse folds the drift of the trace along the dispersion axis into it, so the profile in any
single column is somewhat narrower. Even so, **FWHM ≈ 6 pixels** is the yardstick for everything
that follows. A trace error of half a pixel is negligible against a six-pixel profile; an error
of four pixels is catastrophic. It also sets the scale for what "accurate enough" means: there is
no point chasing hundredths of a pixel when the source itself is six across.

Now the crucial question: does the source stay at row 135? Here are three cross-dispersion cuts,
taken near the left edge, the middle, and the right edge of the detector.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)

for x0 in (32, 512, 960):
    cut = obj.data[:, x0 - 16 : x0 + 16].mean(axis=1)
    ax.plot(
        rows, cut / cut[np.abs(rows - 135) <= 10].max(), lw=0.9, label=f"column {x0}"
    )

ax.axvline(135, c="0.6", lw=0.8, ls="--")
ax.legend(frameon=False)
ax.set(xlim=(122, 148), xlabel="Cross-dispersion [pix]", ylabel="Normalized signal");

The peak is not in the same place in the three cuts. It drifts by several pixels, comparable to
the width of the profile itself. A fixed row *cannot* follow it.

## `FlatTrace`: assuming a straight line

The simplest possible trace is a constant. `FlatTrace` takes the image and a single
cross-dispersion position, and produces a trace that sits at that row for every column.

In [ ]:
flat_trace = FlatTrace(obj, trace_pos=135)

print(type(flat_trace.trace), flat_trace.trace.shape)
print(flat_trace.trace[:5])

Every `Trace` object exposes the same thing: a `.trace` attribute holding one cross-dispersion
position per dispersion column. That uniform interface is what lets `Background` and the
extraction classes accept any of the three classes interchangeably.

Overplotted on the frame, a flat trace at row 135 looks plausible enough, until you stretch the
vertical axis. The panel below shows only 40 rows around the source, so the frame is drawn
about twenty-five times taller than it really is; the drift that was invisible in Chapter 1 is
obvious here.

In [ ]:
def show_trace(ax, traces, ylim=(115, 155), title=None):
    """Display the science frame zoomed on the source, with traces overplotted."""
    ax.imshow(
        obj.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
    )
    for tr, label, style in traces:
        ax.plot(np.asarray(tr), label=label, **{"lw": 1.0, **style})
    ax.set(
        xlim=(0, nx - 1),
        ylim=ylim,
        xlabel="Dispersion [pix]",
        ylabel="Cross-dispersion [pix]",
    )
    if title:
        ax.set_title(title, size=9)
    if any(label for _, label, _ in traces):
        ax.legend(frameon=False, loc="upper left", fontsize=8)


fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
show_trace(ax, [(flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--"))]);

The ridge starts below the line, rises to meet it around the middle of the detector, and
falls away again to end well below it. `FlatTrace` is not wrong here so much as *insufficient*,
and it is worth being clear about when it is the right call anyway. If your spectrograph really
is well aligned, a flat trace is faster, has no fit to go wrong, and cannot be seduced by a
cosmic ray. It is also the sensible choice for arc frames, where the "source" fills the slit and
there is no ridge to follow at all. Chapter 6 uses exactly that.

## `FitTrace`: fitting a curve

`FitTrace` does the obvious thing, in three steps:

1. **Bin** the image into groups of columns along the dispersion axis, averaging each group.
   Binning trades resolution along the trace for signal-to-noise in each measurement.
2. **Find the peak** in each bin's cross-dispersion profile, using `max`, `centroid`, or
   `gaussian`.
3. **Fit a 1D model** through those peak positions and evaluate it at every column.

Each step has a parameter, and the defaults are not always what you want. The default model is
a `Polynomial1D` of degree 1: a straight line, free to tilt but not to bend. The three cuts in
the previous section showed the peak rising and then falling across the detector, and no line
can follow that. The default is not broken; it does exactly what a degree-1 polynomial can do,
which is not enough here. The question is how much more freedom to give it.

### Choosing the polynomial degree

There is no ground truth to compare against, so the test has to be one of **convergence**: fit a
range of degrees and ask how much the trace *changes* when one more term is added. While the
model is missing real structure, each extra degree moves the trace by a substantial fraction of
a pixel. Once it has enough freedom, an extra term can only follow noise, and the change drops
to the level of the per-bin measurement error and stays there. The statistic below is the RMS
change, over all columns, between the trace fitted at degree $d$ and at degree $d+1$.

The model here is `Chebyshev1D` rather than `Polynomial1D`. Both describe the same family of
curves, but a plain power series in $x$ over a range of 0–1000 produces a badly conditioned fit
at high degree ($x^4$ spans fourteen orders of magnitude across the frame), and `astropy` will
warn about it. Chebyshev polynomials are rescaled to the data's own range and stay well behaved.
For a curve this gentle the fitted results are indistinguishable; the conditioning is free.

In [ ]:
degrees = range(1, 8)
fitted = {}
for degree in degrees:
    trace = FitTrace(
        obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(degree)
    )
    fitted[degree] = np.asarray(trace.trace)

change = {d: np.std(fitted[d + 1] - fitted[d]) for d in degrees[:-1]}

for d, c in change.items():
    print(
        f"degree {d} -> {d + 1}:  span {np.ptp(fitted[d]):.2f} pix,  "
        f"RMS change {c:.3f} pix"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)

ax.plot(list(change), list(change.values()), "o-", lw=0.9, ms=4)
ax.axhline(0.1, c="0.6", lw=0.8, ls="--")
ax.text(6, 0.11, "0.1 pix", ha="right", size=8, c="0.4")
ax.set(
    yscale="log",
    xlabel="Polynomial degree $d$",
    ylabel="RMS change, $d \\to d+1$ [pix]",
);

The curve tells a clear story. Going from degree 1 to 2 moves the trace by 0.78 pixels RMS, an
eighth of the profile's FWHM. The trace is fundamentally curved, and the quadratic term does
nearly all the work. Degrees 3 and 4 each still shift it by about 0.05 pixels, small but
genuine corrections. After that the changes collapse: degree 5 moves the trace by 0.015 pixels,
and degrees 6 and 7 by a few thousandths, which is nothing against a six-pixel profile. The fit
has converged.

So degree 4, though degree 3 would be defensible too, since the correction degree 4 adds is
0.05 pixels, far below the ~0.1-pixel level where trace error starts to matter against a
six-pixel profile. The general rule is to take the **lowest degree at which adding a term stops
changing the trace**. Every extra free parameter is another chance for the fit to chase noise,
or to swing wildly past the ends of the data where nothing constrains it.


In [ ]:
trace = FitTrace(
    obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
trace_y = np.asarray(trace.trace)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "FitTrace, degree 4", dict(c="C1")),
        (flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--", lw=0.8)),
    ],
)

print(f"span {np.ptp(trace_y):.2f} pix:  {trace_y[0]:.2f} -> {trace_y[-1]:.2f}")

The fitted trace follows the ridge along the whole detector: it starts 1.7 pixels below the
flat line, touches it near the middle, and drops 3.6 pixels below it by the red end, a total
drift of 3.7 pixels.


## Choosing `peak_method`

`FitTrace` offers three ways to locate the peak within each bin, and they are not
interchangeable. The comparison below runs all three at degree 4, with and without a `window`,
and reports each trace's span and its RMS difference from the `gaussian` trace adopted above.
The point is not to crown a winner, since without ground truth the reference is a choice, but to
see which methods agree and which one goes badly wrong.


In [ ]:
print(f"{'peak_method':>12}  {'window':>7}  {'span [pix]':>11}  {'RMS diff [pix]':>15}")
for method in ("max", "centroid", "gaussian"):
    for window in (None, 20):
        tr = np.asarray(
            FitTrace(
                obj,
                bins=32,
                peak_method=method,
                guess=135,
                window=window,
                trace_model=models.Chebyshev1D(4),
            ).trace
        )
        r = np.std(tr - trace_y)
        print(f"{method:>12}  {str(window):>7}  {np.ptp(tr):>11.2f}  {r:>15.3f}")

The obvious failure is `centroid` without a `window`. It reports a span of 22.6 pixels where
every other method finds 3.5–3.7, and its trace sits 6.3 pixels RMS from the others. The reason
is that the centroid is taken over the *entire* cross-dispersion axis: sky on both sides and the
faint star at row 311 pull it toward the middle of the slit, and at the red end, where the target
is faintest, they drag it twenty rows off the source. Add `window=20` and it falls into line,
within 0.05 pixels of the `gaussian` trace. Note the failure mode: no crash, no warning, only a
trace that quietly leaves the source. `max` and `gaussian` are peak-local by construction, so
the window makes no difference to them.

`max` agrees with `gaussian` to 0.08 pixels RMS. That closeness is misleading, though, because
it hides what `max` actually does. Look at the individual bin measurements rather than the
fitted trace:


In [ ]:
edges = np.linspace(0, nx, 33, dtype=int)
centers = 0.5 * (edges[:-1] + edges[1:])
near = np.abs(rows - 135) <= 8

max_peaks = np.array(
    [
        rows[near][obj.data[near, edges[i] : edges[i + 1]].mean(axis=1).argmax()]
        for i in range(32)
    ]
)
scatter = np.std(max_peaks - np.interp(centers, np.arange(nx), trace_y))

print("first eight `max` peaks:", max_peaks[:8])
print(f"all integers: {np.allclose(max_peaks, np.round(max_peaks))}")
print(f"scatter about the fitted trace:  {scatter:.3f} pix")
print(f"quantization floor, 1/sqrt(12):  {1 / np.sqrt(12):.3f} pix")

`max` returns the index of the brightest pixel, so every measurement is a whole number, and its
scatter about the smooth fitted trace is 0.25 pixels, essentially the floor imposed by rounding
to integers, which for uniformly distributed rounding error is $1/\sqrt{12} \approx 0.29$ pixels.
`max` is doing as well as `max` can possibly do.

The reason the *fitted* `max` trace nevertheless lands within 0.08 pixels of the `gaussian`
one is that the polynomial averages 32 such measurements, and independent rounding errors
partly cancel. That is a real effect and worth relying on, but it depends on having plenty of
bins and a stiff model. Cut to eight bins, or allow a high-degree curve, and the quantization
stops averaging away and starts being fitted. This is the general shape of the trade-off: **more
bins average down the noise in each measurement, fewer bins put more signal into each one.**

So: reach for `gaussian` by default on a bright, well-formed profile, and `max` when speed
matters or the profile is too ugly to fit. We will see shortly that on a genuinely faint source
the preference inverts.

:::{note}
`guess` and `window` are worth setting even when you do not strictly need them. `guess` tells
`FitTrace` where to start instead of taking the brightest row of the collapsed frame; `window`
confines the search to `guess ± window`. Together they make the fit deterministic rather than
dependent on whatever happens to be brightest.
:::

## `ArrayTrace`: supplying positions directly

Sometimes no polynomial will do. The trace may have a kink, the instrument may have flexed
mid-exposure, or the positions may come from somewhere else entirely: a previous frame in a
time series, a model of atmospheric dispersion, a trace measured on a brighter star in the same slit.
`ArrayTrace` takes the positions verbatim.

It needs one value per dispersion column. The simplest demonstration is a round trip (hand it
the positions of the trace fitted above and get the same trace back), but the realistic use on
a dataset like this one is a time series: fit the trace once, on a bright exposure, and apply it
to every other frame through `ArrayTrace`, so that all of them are extracted along exactly the
same path. `ArrayTrace` also fixes up mismatched lengths rather than refusing them, and it is
worth seeing what that does:

In [ ]:
array_trace = ArrayTrace(obj, trace=trace_y)
print(f"round trip exact: {np.allclose(array_trace.trace, trace_y)}")

short_trace = ArrayTrace(obj, trace=trace_y[:900])
n_masked = np.ma.getmaskarray(short_trace.trace).sum()
print(
    f"900 positions in -> {short_trace.trace.shape[0]} out, {n_masked} of them masked"
)

An array longer than the image is truncated; a shorter one is padded at the end with masked
values, and non-finite entries are masked too. Convenient, but it means a length bug produces a
silently part-masked trace instead of an exception. Check `np.ma.getmaskarray(trace.trace).sum()`
if a trace behaves oddly.

`ArrayTrace` is the more flexible option and correspondingly the more dangerous one: with no
smooth model to enforce, whatever noise is in the input positions goes straight into the trace
and from there into the extracted spectrum. Use it when you genuinely need the freedom, and
smooth the input when you do.

## Faint and crowded sources

Everything so far has traced the brightest thing on the slit. That is what `FitTrace` does by
default: with no `guess`, it collapses the frame and takes the peak row. When the source you
want is *not* the brightest, you have to say so.

The faint star at row 311 peaks at about three percent of the target, and three arguments bring
it within reach. `guess` points the search at the right row, and `window` confines the peak
finding to that many pixels on either side of it, so the bright target cannot capture the fit.
`disp_bounds=(lo, hi)` restricts the fit along the *dispersion* axis instead: the bins are laid
over that column range only, the model is fitted to the peaks found there, and it is then
evaluated at every column of the image, so outside the bounds the trace is an extrapolation.
This matters for a source whose signal varies along the trace, and this one fades toward the
red: its flux in a five-pixel aperture drops from about 6000 e- per column mid-detector to
2000 e- in the last quarter, so bounding the fit to the first 700 columns keeps the poorly
measured bins out. Fewer bins and a lower-degree model complete the recipe, since each bin holds
less signal and the fit has fewer trustworthy points. Which `peak_method` to use is less obvious
than on the bright source, so compare:

In [ ]:
print(
    f"{'disp_bounds':>12}  {'peak_method':>11}  {'span [pix]':>10}  {'red end [pix]':>13}"
)
faint_star_traces = {}
for bounds in (None, (0, 700)):
    for method in ("max", "centroid", "gaussian"):
        tr = FitTrace(
            obj,
            bins=16,
            guess=311,
            window=20,
            peak_method=method,
            trace_model=models.Chebyshev1D(2),
            disp_bounds=bounds,
        )
        y = faint_star_traces[bounds, method] = np.asarray(tr.trace)
        print(f"{str(bounds):>12}  {method:>11}  {np.ptp(y):>10.2f}  {y[-1]:>13.2f}")

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=40_000),
)
ax.axvline(700, c="0.4", lw=0.8, ls=":")
for key, color, label in (
    ((None, "max"), "C1", '"max", full range'),
    ((None, "gaussian"), "C3", '"gaussian", full range'),
    (((0, 700), "gaussian"), "C0", '"gaussian", disp_bounds=(0, 700)'),
):
    ax.plot(faint_star_traces[key], c=color, lw=1.0, label=label)
ax.legend(frameon=False, loc="lower left", fontsize=8)
ax.set(
    xlim=(0, nx - 1),
    ylim=(298, 324),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
);

**The methods no longer agree.** On the bright target all three, given a window, landed within
a tenth of a pixel RMS of each other. Over the full range their spans now differ by a factor of
three, and the image says which to believe. `max` runs down the ridge. `centroid` is flattened:
on a source only a few times the sky level, the sky inside the ±20-pixel window drags every
measurement toward the window's center at row 311, so the drift comes out at half its size.
`gaussian` leaves the ridge by more than a pixel at the red end, where the star is faintest and
the per-bin Gaussian fits are no longer stable.

`disp_bounds` sorts this out for `gaussian`, and only for `gaussian`. Confined to the first 700
columns, its span settles at 1.7 pixels, hardly depends on where the bound is placed (600 to 800
all give the same span to 0.05 pixels), and its extrapolated red end lands within a third of a
pixel of the full-range `max` trace, so two independent routes agree even where neither measured
the star well. `max`, by contrast, comes back perfectly flat: within those 700 columns the star
moves by less than half a pixel, so every one of its integer bin peaks is row 311. `max` is
limited by quantization, not by signal, and narrowing the fit only shortens its lever arm. Keep
in mind that the extrapolated part of a bounded trace is a model assumption, sound for a smooth
optical distortion and wrong for a kink or a flexure jump, so draw it over the image before
trusting it.

The lesson: **`gaussian` is the most precise method when the profile is well measured and the
most fragile when it is not**, and "not" means faint, blended, or oddly shaped profiles, exactly
the cases where you most want the precision. Either fall back on crude, unconditionally robust
`max`, or keep `gaussian` and confine the fit with `disp_bounds` to where the profile is well
measured. Here the two agree to a twentieth of the FWHM; chapters 7 and 8 use `max` over the
full range.

`window` cuts both ways, too. It keeps the fit from wandering onto a brighter neighbor, but it
also caps how far the trace can move: a source that drifts further than `window` will hit the
edge of its own search region and flatten out. Set it comfortably wider than the drift you
expect, and narrower than the separation to the nearest interloper.

## Shifting a trace

`Trace` objects support addition and subtraction, returning **shifted copies** displaced along
the cross-dispersion axis. The original is untouched.

In [ ]:
above = trace + 20
below = trace - 20

print(f"original  {np.asarray(trace.trace)[512]:.2f}")
print(f"+20       {np.asarray(above.trace)[512]:.2f}")
print(f"-20       {np.asarray(below.trace)[512]:.2f}")

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "trace", dict(c="C1")),
        (above.trace, "trace + 20", dict(c="C0", ls="--", lw=0.8)),
        (below.trace, "trace - 20", dict(c="C0", ls="--", lw=0.8)),
    ],
    ylim=(100, 170),
);

This is not a curiosity. It is how the next chapter places its sky windows: the background is
measured in bands that run *parallel to the trace*, offset either side of the source, so that
every pixel in a band is at the same distance from the object as the frame curves. Offsetting a
straight line instead would let the sky bands slide relative to the source, which is precisely
the error a fitted trace exists to avoid.

Positions that fall outside the image are masked automatically rather than raising, so a shift
that runs off the detector degrades gracefully.

## Summary

- A trace is one cross-dispersion position per dispersion column, exposed as `.trace` on all
  three classes. Whatever produced it, downstream operations consume it identically.
- On this frame the source moves **3.7 pixels** across the detector, against a profile FWHM of
  **6 pixels**, a drift a straight trace cannot follow.
- Validate with what you have: scan the degree and stop at the lowest one where adding a term no
  longer moves the trace (degree 4 here), compare peak methods against each other, and draw the
  result over the image.
- `peak_method` depends on the source. On a bright trace `gaussian` and `max` agree to a tenth
  of a pixel, though `max` quantizes each bin measurement at $1/\sqrt{12}$ pixels and relies on
  the fit to average that away; `centroid` silently wanders off the source unless given a
  `window`.
- Faint or crowded sources need `guess` and `window` to find the right source, fewer bins and a
  lower degree, and `disp_bounds` to confine the fit to the columns with enough signal.
  `gaussian` is fragile where the star fades and `disp_bounds` restores it; `max` is robust over
  the full range but limited by quantization, so bounding cannot help it. Here the two routes
  agree to a twentieth of the FWHM.
- `trace ± delta` gives shifted copies, which is how background windows are placed.

Next: [Arc Lamps](05-arc-lamps.ipynb), the wavelength-reference frames, and the first place the
frame's tilt becomes impossible to ignore.